# CuBench — Full Grid, Colab Runner

This notebook runs the **entire CuBench pipeline** end-to-end on Google Colab CPU
runtime, to reach 100% completion of the experimental grid that ran out of local
throughput/time budget on the author's machine (see `STATUS.md`, search
"LOCAL GRID TRAINING STOPPED"). Local partial results already proved the pipeline
correct (leakage tests green, real bugs found and fixed at every phase); this
notebook's job is to **finish the grid**, not to redesign anything.

**Companion doc:** `docs/cubench_colab_runbook.md` — read it first if this is your
first run. It covers cell order, expected runtime, resume behavior, and how to get
results back into `D:\copper`.

**Governing decision (plan D12):** CPU-only, no GPU needed anywhere in this pipeline.
That removes the GPU-disconnect checkpoint-resume problem class entirely — there is no
CUDA state to lose. What remains is Colab's wall-clock session limit (~12h, often
less on free tier) and the possibility of a plain disconnect. The design below answers
that with the **same mechanism the local run already used and already proved**:
every grid cell (`model, target, horizon, fold, seed`) is appended as one JSON line to
`results/cubench/grid_results.jsonl` **immediately** after it finishes, and the runner
reads that file at start-up to skip anything already done. A disconnect loses at most
the in-flight cell, never the run. On top of that, this notebook periodically copies
the file to Drive so a full session loss (not just a crash) also survives.

**Sections:**
1. Setup (Drive mount, repo clone, dependency install)
2. Data acquisition (+ FRED coverage assertions)
3. Feature engineering (+ shape assertion)
4. Leakage tests (hard gate)
5. Full grid training (nulls -> econometric -> linear -> trees -> deep), incremental
   save, Drive sync, resume, progress checker
6. Full statistical evaluation
7. Full ablations (A0-A7)
8. Figures
9. Package results for download / sync back to `D:\copper`


## 1. Setup

### 1.1 Why Drive, and why *not* the old checkpoint-resume machinery

Paper 1's Colab notebook (`notebooks/archive_paper1_vmd_mfgnn/vmd_mfgnn_v2_colab.ipynb`)
built a fairly heavy checkpoint/resume system because it had to survive **GPU
disconnects mid-epoch** — losing a partially-trained network is expensive, and Paper 1
also had a real, documented incident where a **file-size-based Drive-sync staleness
check** silently kept a stale cached file instead of a fresh one (`docs/todo.txt` item
5) and corrupted a run.

CuBench does not have that risk class:
- **No GPU checkpoints.** Everything is CPU, and the only "state" worth preserving is
  *which grid cells have already produced a result row* — that's a one-line JSON
  append per cell, not a multi-megabyte model checkpoint.
- **No file-size staleness check anywhere in this notebook.** Drive sync here is a
  plain `cp`/`shutil.copy` of a small, append-only `.jsonl` file, done by *content*
  (we always overwrite Drive with the current local file — there is no "is this newer"
  heuristic to get wrong).

So the pattern used below is deliberately simpler than Paper 1's: **local disk is the
working copy** (fast, no Drive I/O per row), and **Drive is a periodic, cheap backup**
of the one file that matters (`grid_results.jsonl`) plus the evaluation/figure
outputs at the end. If Colab disconnects entirely, you re-mount Drive, copy the backed
up `grid_results.jsonl` back into the fresh clone's `results/cubench/`, and re-run —
the resume logic (identical code, `src/cubench/walkforward.py::run_grid`) picks up
exactly where it left off.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/cubench_backup'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive backup directory:', DRIVE_DIR)


In [ ]:
# Clone the repo. This notebook is packaged as part of the repo itself
# (notebooks/cubench_colab.ipynb) but Colab starts from a bare VM, so we clone fresh.
# Repo remote (verified via `git remote -v` on the dev machine):
#   https://github.com/anmol0705/Copper_Price_Forecasting.git
# IMPORTANT: this repo's default remote branch may not be the branch this work was
# developed on (`graph-fix-experiment`). Set BRANCH below to whatever branch/tag holds
# the CuBench code (src/cubench/, scripts/cubench_*.py, configs/cubench.yaml) if you
# are not sure it has been merged to main.
import os

REPO_URL = 'https://github.com/anmol0705/Copper_Price_Forecasting.git'
BRANCH = 'graph-fix-experiment'  # change if this has since been merged to main
REPO_DIR = '/content/copper'

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print('Repo already present, pulling latest...')
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!git log -1 --oneline
!ls src/cubench scripts | grep cubench


**If the clone above fails** (private repo, no internet egress to GitHub from
your Colab environment, or the branch doesn't exist on the remote): instead upload the
repo as a zip. In a local terminal:

```
cd D:\copper
git archive --format=zip -o cubench_repo.zip HEAD
```

Then in Colab: Files pane -> upload `cubench_repo.zip` -> in a code cell:
```python
import zipfile
os.makedirs('/content/copper', exist_ok=True)
with zipfile.ZipFile('/content/cubench_repo.zip') as z:
    z.extractall('/content/copper')
%cd /content/copper
```
Either path lands you in the same place: a full working copy of the repo at
`/content/copper` with `src/cubench/`, `scripts/cubench_*.py`, `configs/cubench.yaml`,
and `tests/` present.


In [ ]:
# Sanity check: the real CuBench code is present and importable before we spend any
# time on installs.
import sys
sys.path.insert(0, '/content/copper')
required_files = [
    'src/cubench/data.py', 'src/cubench/features.py', 'src/cubench/walkforward.py',
    'src/cubench/models/nulls.py', 'src/cubench/models/econometric.py',
    'src/cubench/models/linear.py', 'src/cubench/models/trees.py', 'src/cubench/models/deep.py',
    'src/cubench/stats_tests.py', 'src/cubench/diagnostics.py', 'src/cubench/backtest.py',
    'src/cubench/ablations.py', 'src/cubench/regimes.py', 'src/cubench/metrics.py',
    'scripts/cubench_build_data.py', 'scripts/cubench_build_features.py',
    'scripts/cubench_run_grid.py', 'scripts/cubench_evaluate.py',
    'scripts/cubench_make_figures.py', 'configs/cubench.yaml', 'requirements_cubench.txt',
    'tests/test_leakage.py',
]
missing = [f for f in required_files if not os.path.exists(f)]
assert not missing, f'Missing required files, clone/upload is incomplete: {missing}'
print(f'All {len(required_files)} required files present.')


### 1.2 Install dependencies

`requirements_cubench.txt` documents a real, non-obvious install hazard found on the
dev machine: **`pypbo` is not pip-installable** — the `esvhd/pypbo` GitHub repo at the
pinned commit ships no `setup.py`/`pyproject.toml` at all, so `pip install
git+https://...` fails outright (not a transient error). The working fix used locally
was to clone the pinned commit and copy the `pypbo/` source directory straight into
`site-packages`. We replicate that exact approach here rather than trust
`pip install git+...` to work on Colab (it will fail for the identical structural
reason — nothing about being on Colab changes what files are in that repo).


In [ ]:
%%time
# Base scientific stack: Colab already ships numpy/pandas/scipy/scikit-learn/statsmodels/
# matplotlib/seaborn. We only need the CuBench-specific additions.
!pip -q install lightgbm>=4.5 xgboost>=2.1 "catboost>=1.2" "arch>=7.0" "shap>=0.46" \
    "dieboldmariano>=1.0" "tabulate" pyyaml pyarrow

# torch: Colab's default runtime may already have a CUDA build installed; per plan D12
# this pipeline is CPU-only, and importing torch's default install is fine either way
# since none of the CuBench deep models call .cuda(). We do not force a CPU-only wheel
# reinstall (that would be a slow, unnecessary download) -- we just never move tensors
# to a GPU device anywhere in src/cubench/models/deep.py (verified: that file has no
# .cuda()/.to('cuda') call).
import importlib
if importlib.util.find_spec('torch') is None:
    !pip -q install torch --index-url https://download.pytorch.org/whl/cpu
import torch
print('torch', torch.__version__, '| cuda available (unused by this pipeline):', torch.cuda.is_available())


In [ ]:
%%time
# pypbo: replicate the exact working install procedure documented in
# requirements_cubench.txt (plain pip install fails -- the repo has no setup.py).
PYPBO_SHA = '4d723f06498267a2a6280cb9d7d5649348e961d1'
import subprocess, shutil, site

!rm -rf /content/pypbo_pin
!git clone -q https://github.com/esvhd/pypbo.git /content/pypbo_pin
!cd /content/pypbo_pin && git checkout -q {PYPBO_SHA}

site_packages = site.getsitepackages()[0]
dest = os.path.join(site_packages, 'pypbo')
if os.path.isdir(dest):
    shutil.rmtree(dest)
shutil.copytree('/content/pypbo_pin/pypbo', dest)

import pypbo
print('pypbo importable from', pypbo.__file__)


In [ ]:
# Final import smoke test -- every package src/cubench actually imports, verified
# importable before we touch any real logic. This is the Colab-environment equivalent
# of the Week-A1 "smoke-test import lightgbm, xgboost, catboost, arch, shap, pypbo" step.
import lightgbm, xgboost, catboost, arch, shap, pypbo, dieboldmariano, yaml, tabulate
print('All CuBench dependencies import cleanly.')
print('lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__,
      '| catboost', catboost.__version__)


## 2. Data acquisition

Runs the real downloader (`src/cubench/data.py`), unmodified. It shells out to
`curl` for every FRED pull rather than using `requests`/`fredapi`/`pandas_datareader` —
this is not a Colab-specific choice, it's carried forward from a **verified** local
finding (`requests` reliably times out against `fred.stlouisfed.org` on the dev
machine's network stack; `curl` against the identical URL succeeds). Colab's network
environment is different and may not hit that particular TLS/schannel issue at all,
but there is no reason to gamble a multi-hour run on an unproven code path when a
proven one exists — we keep curl and verify it works here, early, as its own cell,
so any Colab-specific network failure (egress blocked, DNS, etc.) surfaces in seconds,
not hours into the grid run.


In [ ]:
# Early network sanity check -- fail fast if Colab's egress can't reach FRED or Yahoo.
!curl -s -o /dev/null -w "FRED curl HTTP status: %{http_code}\n" "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFII10"
import yfinance as yf
_test = yf.Ticker('HG=F').history(period='5d')
print('yfinance smoke test rows:', len(_test))
assert len(_test) > 0, 'yfinance returned no rows -- Colab network/yfinance issue, investigate before proceeding'


In [ ]:
%%time
# Real data pull -- unmodified script, same one used locally.
!python scripts/cubench_build_data.py


In [ ]:
# Sanity cell mirroring tests/test_leakage.py::test_fred_coverage_assertions -- run
# BEFORE the full leakage suite (section 4) so a coverage regression on FRED's serving
# side is caught immediately, not after a long feature-build.
import pandas as pd

EXPECTED_FIRST_OBS = {
    'fred_real_yield_10y.csv': '2010-01-05',      # DFII10
    'fred_baa_credit_spread.csv': '2010-01-05',   # BAA10Y
    'fred_ppi_all_commodities.csv': '2009-01-01', # PPIACO
    'fred_industrial_production.csv': '2009-01-01', # INDPRO
}
RAW_DIR = 'data/cubench/raw'
for fname, expected_by in EXPECTED_FIRST_OBS.items():
    path = os.path.join(RAW_DIR, fname)
    df = pd.read_csv(path, parse_dates=['date'])
    first = df['date'].min()
    ok = first <= pd.Timestamp(expected_by)
    print(f'{fname}: first_obs={first.date()} (expected <= {expected_by}) -> {"OK" if ok else "REGRESSION"}')
    assert ok, f'{fname} coverage regressed -- FRED is now serving less history than expected. See plan D4/R2.'
print('All FRED coverage assertions passed.')


## 3. Feature engineering

In [ ]:
%%time
# Real feature-build script, unmodified -- Blocks A/B'/C/D + T1/T2/T3 targets, folds.json.
!python scripts/cubench_build_features.py


In [ ]:
# Known-good shape assertion -- this is the exact result verified locally
# (data/cubench/features.parquet, confirmed 2026-08-22 via .venv_corr): 4023 rows x 81
# columns, 2010-01-04 -> 2025-12-30. A mismatch here means something in the data or
# feature pipeline behaved differently on Colab and must be investigated before any
# model trains on it -- do not proceed past a failed assertion in this cell.
import pandas as pd
df = pd.read_parquet('data/cubench/features.parquet')
print('shape:', df.shape)
print('date range:', df['date'].min(), '->', df['date'].max())

assert df.shape == (4023, 81), f'UNEXPECTED SHAPE {df.shape}, expected (4023, 81) -- investigate before proceeding'
assert str(df['date'].min().date()) == '2010-01-04'
assert str(df['date'].max().date()) == '2025-12-30'
print('Feature matrix shape/date-range assertions PASSED.')


## 4. Leakage tests — hard gate

Per this project's standing discipline, **no model may train on out-of-sample
data until `tests/test_leakage.py` passes in full**, in this environment (not just
"it passed once on the dev machine"). Locally this suite takes ~6 minutes
(`10 passed, 34 warnings in 361.31s`, per `STATUS.md`); expect similar or somewhat
different timing on Colab's CPU. **If any test fails here, STOP** — do not proceed to
Section 5. A failure means either (a) a genuine environment-dependent leakage bug
(different pandas/numpy float precision, different `merge_asof` behavior across
versions, etc.) or (b) a real bug the local run never exercised. Either way, silently
continuing past a failed leakage gate is exactly the failure mode this project's whole
discipline exists to prevent.


In [ ]:
%%time
result = os.system('python -m pytest tests/test_leakage.py -v')
assert result == 0, (
    'LEAKAGE TESTS FAILED in this Colab environment. Do not proceed to grid training. '
    'Investigate the pytest output above -- this may be a genuine environment-dependent '
    'leakage bug (see docs/cubench_colab_runbook.md, "If a section fails").'
)
print('Leakage gate PASSED -- safe to proceed to grid training.')


## 5. Full grid training

This is the long-running core of the notebook. **Model families run in this
order, fast/verified-first:**

1. `nulls` (5 models) -- already 100% complete locally; should reproduce near-instantly
   here and is a cheap end-to-end sanity check that the whole harness works in Colab.
2. `econometric` (5 models: har_rv, har_rv_q, garch11, gjr_garch, arima) -- 100% done
   locally, same sanity-check role, GARCH refits are the slowest part of this tier.
3. `linear` (elasticnet) -- 100% done locally, fast.
4. `trees` (lgbm, xgboost, catboost, randomforest) -- **the bottleneck.** Locally,
   before the machine ran out of throughput: lgbm ~66% done, xgboost ~30%,
   catboost ~34%, randomforest ~31% (of the full 3 targets x 3 horizons x 11 folds x 5
   seeds = 495 cells each). Concurrent CPU contention from running multiple tree
   families in parallel was diagnosed locally as the actual throughput ceiling (Phase
   3) -- **this notebook runs tree families sequentially, one at a time**, specifically
   to avoid re-creating that contention on Colab's (typically 2-core) standard runtime.
5. `deep` (lstm, transformer) -- essentially unstarted locally (lstm 1 cell,
   transformer 0, against an already-reduced target). **Read the scope-note callout
   below before running this cell** -- the deep-learning grid is NOT the full 3x3
   target/horizon grid the plan's Section 3 headline describes, and this notebook
   cannot change that without editing `src/cubench/models/deep.py`, which Phase 6 is
   not authorized to do.

**Realistic wall-clock estimates** (based on locally-observed throughput; Colab's CPU
is typically comparable to a modern laptop core, occasionally slower on the free
tier's shared/throttled instances -- these are honest planning numbers, not
optimistic ones):
- nulls + econometric + linear: **10-30 minutes** total (already fast locally, these
  are validating quantities, not the compute-heavy tier).
- trees (all 4 families, run sequentially, full grid, 5 seeds): **4-10 hours**, the
  single largest time cost in the whole notebook. LightGBM is the fastest of the four;
  catboost has historically been the slowest per-cell locally.
- deep (lstm + transformer, restricted grid per the scope note below): **1-3 hours**
  for the ~66-cell-per-model subset the code actually runs (not the full 594-cell grid
  the plan's model-roster table would imply).

**Total realistic estimate for Section 5: 6-14 hours of CPU time**, which will almost
certainly span more than one Colab session on the free tier. This is exactly why the
incremental-save + Drive-sync + resume design exists: run what you can in one session,
let it disconnect, come back, re-run the cells below, and it picks up where it left
off. Do not treat a disconnect mid-grid as a failure -- it is the expected mode of
operation for a run this long on a free/standard Colab tier.


In [ ]:
# ---- Drive sync helpers, used throughout Section 5 ----
import shutil, time as _time

JSONL_LOCAL = 'results/cubench/grid_results.jsonl'
JSONL_DRIVE = os.path.join(DRIVE_DIR, 'grid_results.jsonl')

def sync_to_drive(note=''):
    if os.path.exists(JSONL_LOCAL):
        shutil.copy2(JSONL_LOCAL, JSONL_DRIVE)
        n = sum(1 for _ in open(JSONL_LOCAL))
        print(f'[{_time.strftime("%H:%M:%S")}] synced {n} lines to Drive {note}')
    else:
        print('no grid_results.jsonl yet, nothing to sync')

def restore_from_drive_if_present():
    os.makedirs('results/cubench', exist_ok=True)
    if os.path.exists(JSONL_DRIVE) and not os.path.exists(JSONL_LOCAL):
        shutil.copy2(JSONL_DRIVE, JSONL_LOCAL)
        n = sum(1 for _ in open(JSONL_LOCAL))
        print(f'RESTORED {n} lines from Drive backup (fresh clone / new session detected)')
    elif os.path.exists(JSONL_DRIVE) and os.path.exists(JSONL_LOCAL):
        # Both exist (e.g. re-running restore after a partial local run in this same
        # session) -- merge by unioning lines, since run_grid's own dedup keys on
        # (model,target,horizon,fold,seed) and tolerates duplicate lines safely
        # (cubench_aggregate.py dedups by key, last-line-wins). We do NOT pick "newer
        # by file size/mtime" (that heuristic is the exact bug class documented in
        # docs/todo.txt item 5) -- we simply concatenate both sources; downstream
        # aggregation is idempotent to duplicates by construction.
        with open(JSONL_DRIVE) as f:
            drive_lines = f.readlines()
        with open(JSONL_LOCAL, 'a') as f:
            f.writelines(drive_lines)
        print(f'MERGED {len(drive_lines)} lines from Drive backup into local file (dedup happens at aggregation time)')

restore_from_drive_if_present()


In [ ]:
# ---- Progress checker -- re-run this cell any time to see current grid completion,
# no need to understand the underlying jsonl format. ----
import json as _json

def grid_progress():
    path = 'results/cubench/grid_results.jsonl'
    if not os.path.exists(path):
        print('No results yet.')
        return
    seen, errors = {}, 0
    with open(path) as f:
        for line in f:
            try:
                r = _json.loads(line)
            except Exception:
                continue
            key = (r.get('model'), r.get('target'), r.get('horizon'), r.get('fold'), r.get('seed'))
            seen[key] = r
            if 'error' in r:
                errors += 1
    print(f'Total unique completed cells: {len(seen)}  (errored cells: {errors})')
    import collections
    by_model = collections.Counter(k[0] for k in seen)
    for model, n in sorted(by_model.items()):
        print(f'  {model:16s} {n:5d} cells')
    return seen

_ = grid_progress()


In [ ]:
%%time
# ---- 5a. Nulls (fast sanity check) ----
!python scripts/cubench_run_grid.py --family nulls
sync_to_drive('after nulls')


In [ ]:
%%time
# ---- 5b. Econometric (fast sanity check; GARCH refits are the slow part) ----
!python scripts/cubench_run_grid.py --family econometric
sync_to_drive('after econometric')


In [ ]:
%%time
# ---- 5c. Linear (elasticnet) ----
!python scripts/cubench_run_grid.py --family linear
sync_to_drive('after linear')


### 5d. Trees — the bottleneck

Run **one family at a time, sequentially** (do not put these in separate cells and run
them concurrently via Colab's "run in background" — the incremental-jsonl-append
resume logic was verified locally to be race-condition-safe *within* a single
process's dedup check at start-up, but two processes racing to append to the same file
can both start the same cell before either's result is flushed, producing duplicate
(harmless, since aggregation dedups by key, but wasteful) rows. Sequential is simpler
and was the mode actually validated.)

If a session disconnects mid-family, just re-run the same cell after reconnecting —
`restore_from_drive_if_present()` above brings back whatever was synced, and
`cubench_run_grid.py`'s own dedup (via `walkforward.run_grid`) skips everything
already in the file.


In [ ]:
%%time
!python scripts/cubench_run_grid.py --family trees --model lgbm
sync_to_drive('after lgbm')
grid_progress()


In [ ]:
%%time
!python scripts/cubench_run_grid.py --family trees --model xgboost
sync_to_drive('after xgboost')
grid_progress()


In [ ]:
%%time
!python scripts/cubench_run_grid.py --family trees --model catboost
sync_to_drive('after catboost')
grid_progress()


In [ ]:
%%time
!python scripts/cubench_run_grid.py --family trees --model randomforest
sync_to_drive('after randomforest')
grid_progress()


### 5e. Deep learning — read this before running

**Discrepancy from the plan, found while packaging (reported per Phase 6 instructions,
not silently fixed):** the plan's Section 3 model roster describes 12 model families
each run across "3 targets x 3 horizons". The actual shipped
`src/cubench/models/deep.py` and `scripts/cubench_run_grid.py` restrict the deep tier
(lstm, transformer) to **targets {t1, t3} and horizon h=1 only** (T2 quantile deep
learning and h in {5,22} are skipped), with a documented in-code rationale (full grid
would be 594 NN trainings, judged infeasible in one local session). Separately, the
per-model training budget was also cut from the plan's `epochs=100, patience=20` down
to `MAX_EPOCHS=40, PATIENCE=8`, again documented in-code as a response to local
CPU contention from concurrently-running tree families.

Both of these are **hardcoded in `src/cubench/models/deep.py`**, which this phase is
not authorized to modify (Phase 6 packages existing code, it does not change modeling
decisions). Practically: even on Colab, with much more available wall-clock time and
no concurrent-family contention (since Section 5d already finished sequentially by the
time you reach this cell), **the deep-learning grid this notebook produces will still
only cover t1/t3 x h=1 x 11 folds x 3 seeds = 66 cells per model**, not the full grid
implied by the plan's model-roster table. If full-grid deep learning coverage is
wanted, that requires a deliberate, reviewed code change to `deep.py` (widen `targets`/
`horizons` in `run_deep_grid`'s caller, and reconsider `MAX_EPOCHS`/`PATIENCE` now that
CPU contention is no longer the constraint) — a decision for the user, not something
folded into this notebook.


In [ ]:
%%time
!python scripts/cubench_run_grid.py --family deep --model lstm
sync_to_drive('after lstm')
grid_progress()


In [ ]:
%%time
!python scripts/cubench_run_grid.py --family deep --model transformer
sync_to_drive('after transformer')
grid_progress()


In [ ]:
# ---- Final aggregation: grid_results.jsonl -> all_results.json (dedup, numpy-safe) ----
!python scripts/cubench_aggregate.py
sync_to_drive('after aggregation')
grid_progress()


## 6. Full statistical evaluation

Runs the real Phase-4 evaluation entry point, unmodified. It is explicitly
designed to be re-run against whatever coverage currently exists in
`grid_results.jsonl`/`predictions/*.npy` (see its own docstring), so it is safe to run
this section even if Section 5 was interrupted partway — it just produces a fuller
result the more of the grid is complete. Run it again at the very end (after Section 5
fully completes) to regenerate final, complete numbers.


In [ ]:
%%time
!python scripts/cubench_evaluate.py --ablation-full --ablation-time-budget 21600
sync_to_drive('after evaluate')


**Note on `--ablation-full`:** by default `cubench_evaluate.py` runs a
time-capped ablation *smoke test* (this is what Phase 4 ran locally — see
`STATUS.md` Phase 4/5 notes: "Phase 4 only ran a time-capped smoke test locally").
Section 7 below re-runs the ablation layer explicitly with `--ablation-full`, but we
also pass it here so the very first evaluation pass already attempts full ablations if
time allows within the 6-hour budget given (`--ablation-time-budget 21600` seconds).
Adjust the budget down if you want a faster first pass and plan to rely on Section 7's
dedicated full run instead.


## 7. Full ablations (A0–A7)

Phase 4 locally only ran a **time-capped smoke test** of the ablation grid
(A0-A7 rung definitions, LightGBM, 5 seeds, 3 targets x 3 horizons — see
`docs/cubench_implementation_plan.md` Section 4.6). Colab has materially more time
budget available in a single run than the local session did, so this section
re-invokes evaluation with a generous ablation time budget and **no artificial cap**
beyond what you set here. If Section 5's tree-model grid is not yet fully complete,
the ablation layer (which retrains LightGBM per ablation rung, independent of the main
grid's saved predictions) can still run — it does its own training internally
(`src/cubench/ablations.py::run_ablation_cell`), it does not depend on
`grid_results.jsonl` coverage.


In [ ]:
%%time
# Full A0-A7 ablation grid: LightGBM, 5 seeds, 3 targets x 3 horizons x 8 rungs.
# time_budget_s is generous here (10 hours) -- raise/lower as your session allows.
# If this cell is interrupted by a disconnect, re-running scripts/cubench_evaluate.py
# does NOT resume ablations mid-grid (ablations are not incrementally jsonl-logged the
# way Section 5's main grid is -- they write one results.json at the end of
# run_ablations). Budget your session time for this cell accordingly, or lower
# --ablation-time-budget to fit comfortably within one session.
!python scripts/cubench_evaluate.py --ablation-full --ablation-time-budget 36000 --skip-backtest
sync_to_drive('after full ablations')


**Discrepancy note:** unlike Section 5's grid training, the ablation layer
(`run_ablations` in `scripts/cubench_evaluate.py`, calling
`src/cubench.ablations.run_ablation_cell`) has **no incremental per-cell persistence
and no resume mechanism** — it holds results in memory and writes
`results/cubench/ablations/ablation_results.json` once at the end (or on hitting
`time_budget_s`). A mid-run disconnect during this section loses the in-progress
ablation pass entirely, unlike Section 5 where at most one cell is lost. This is a
property of the existing `src/cubench/ablations.py`/`cubench_evaluate.py` code, which
Phase 6 is not modifying — flagged here so you can plan session length around it (e.g.
run this section in a dedicated, uninterrupted session, or lower the time budget to
something you're confident will complete in one sitting).


## 8. Figures

In [ ]:
%%time
!python scripts/cubench_make_figures.py
sync_to_drive('after figures')


In [ ]:
# Quick visual check that figures were produced with non-trivial (not all-partial)
# data, now that (hopefully) the grid is complete.
import glob
figs = sorted(glob.glob('results/cubench/figures/*.png'))
print(f'{len(figs)} figure PNGs produced:')
for f in figs:
    print(' ', f)
from IPython.display import Image, display
if figs:
    display(Image(figs[0]))


## 9. Package for download

In [ ]:
import shutil

# Full results/cubench/ zip -- predictions, significance, backtest, ablations, regimes,
# figures, jsonl/json aggregates. This is everything needed to reproduce every number
# and figure without retraining (the standing "recompute from raw .npy" practice).
shutil.make_archive('/content/cubench_results', 'zip', 'results/cubench')
print('Packaged: /content/cubench_results.zip')
print('Size (MB):', os.path.getsize('/content/cubench_results.zip') / 1e6)

# Also copy the zip to Drive so it survives even if you don't download it immediately.
shutil.copy2('/content/cubench_results.zip', os.path.join(DRIVE_DIR, 'cubench_results.zip'))
print('Also copied to Drive:', os.path.join(DRIVE_DIR, 'cubench_results.zip'))


In [ ]:
from google.colab import files
files.download('/content/cubench_results.zip')


### Getting results back into `D:\copper`

1. Download `cubench_results.zip` via the cell above (or grab it from
   `MyDrive/cubench_backup/cubench_results.zip` in Google Drive if the direct download
   didn't trigger — Colab sometimes blocks the download prompt).
2. On the local machine, extract it **over** `D:\copper\results\cubench\`, replacing
   the partial local results:
   ```
   powershell -Command "Expand-Archive -Path cubench_results.zip -DestinationPath D:\copper\results\cubench -Force"
   ```
3. Re-run `scripts/cubench_make_figures.py` locally if you want to confirm the figures
   regenerate identically from the downloaded data (they should — same code, same
   inputs).
4. If you want the notebook itself (with its outputs/execution history) copied back
   into the repo for the record: File -> Download -> `.ipynb` in Colab, then replace
   `notebooks/cubench_colab.ipynb` locally and commit.
5. See `docs/cubench_colab_runbook.md` for the full checklist, including what to do if
   only *some* sections completed.
